# Caso 04: VueltaFit — Gimnasios: ¿Qué Plan Deja Más Dinero?
## Día 4 (jueves): ¿Paga el Año y Vuelve el Siguiente? El Hábito y el Control de Robustez

**Objetivo del día:**
Llegar al núcleo causal del problema cruzando la renovación con el comportamiento de las primeras semanas:
1. **La Escalera del Hallazgo:** Medir el impacto de la asistencia en los primeros 21 días sobre la tasa de renovación.
2. **La Primera Reserva:** Evaluar si reservar una clase en los primeros 14 días refuerza el hábito temprano.
3. **El Control de Robustez:** Aislar el efecto del plan controlando por el tramo de asistencia (desmontar la correlación espuria / Paradoja de Simpson).


In [1]:
import duckdb
import pandas as pd
import os

# Inicializar conexión DuckDB y registrar vistas de datos limpios
con = duckdb.connect()
DATA_CLEAN = os.path.join('data', 'clean')

for table in ['socios', 'suscripciones', 'checkins', 'clases', 'reservas', 'pagos', 'sedes']:
    csv_path = os.path.join(DATA_CLEAN, f'{table}_clean.csv').replace('\\', '/')
    con.execute(f"CREATE VIEW v_{table} AS SELECT * FROM read_csv_auto('{csv_path}')")

print('Conexión y vistas DuckDB preparadas.')


Conexión y vistas DuckDB preparadas.


### 1. La Escalera del Hallazgo: Asistencia Temprana (Primeros 21 días)

Contamos los check-ins de cada contrato vencido durante su ventana de bienvenida (`[fecha_inicio, fecha_inicio + 21 días]`):
* **Baja:** 0 a 2 check-ins.
* **Media:** 3 a 5 check-ins.
* **Alta:** 6 o más check-ins ($\ge 6$).


In [2]:
q_escalera = '''
WITH vencidas AS (
    SELECT *
    FROM v_suscripciones
    WHERE TRY_CAST(fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renov AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
),
checkins_21d AS (
    SELECT 
        cr.suscripcion_id,
        COUNT(c.checkin_id) AS n_checkins_21d
    FROM con_renov cr
    LEFT JOIN v_checkins c 
        ON cr.socio_id = c.socio_id
       AND TRY_CAST(c.fecha_hora AS DATE) >= TRY_CAST(cr.fecha_inicio AS DATE)
       AND TRY_CAST(c.fecha_hora AS DATE) <= TRY_CAST(cr.fecha_inicio AS DATE) + INTERVAL 21 DAY
    GROUP BY cr.suscripcion_id
),
datos_completos AS (
    SELECT 
        cr.*,
        COALESCE(ch.n_checkins_21d, 0) AS checkins_21d,
        CASE 
            WHEN COALESCE(ch.n_checkins_21d, 0) <= 2 THEN '1. Baja (0-2)'
            WHEN COALESCE(ch.n_checkins_21d, 0) <= 5 THEN '2. Media (3-5)'
            ELSE '3. Alta (6+)'
        END AS tramo_asistencia
    FROM con_renov cr
    JOIN checkins_21d ch USING (suscripcion_id)
)
SELECT 
    tramo_asistencia,
    COUNT(*) AS total_contratos,
    SUM(CASE WHEN renovo THEN 1 ELSE 0 END) AS total_renovados,
    ROUND(AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END) * 100, 2) AS tasa_renovacion_pct
FROM datos_completos
GROUP BY tramo_asistencia
ORDER BY tramo_asistencia;
'''
df_escalera = con.execute(q_escalera).df()
print('=== La Escalera del Hábito: Renovación por Asistencia Temprana ===')
display(df_escalera)


=== La Escalera del Hábito: Renovación por Asistencia Temprana ===


,tramo_asistencia,total_contratos,total_renovados,tasa_renovacion_pct
0,1. Baja (0-2),1211,267.0,22.05
1,2. Media (3-5),1123,571.0,50.85
2,3. Alta (6+),1423,1119.0,78.64


**Análisis de la Escalera del Hábito (`22,05%` $\rightarrow$ `50,85%` $\rightarrow$ `78,64%`):**
* El número de visitas en los primeros 21 días ordena de forma fulminante la probabilidad de renovar al final del contrato:
  * **1. Baja asistencia (`0–2` visitas en 21 días):** `1.211` contratos ($32,23\%$ del total) $\rightarrow$ solo renuevan **`267` ($22,05\%$)**.
  * **2. Media asistencia (`3–5` visitas en 21 días):** `1.123` contratos ($29,89\%$ del total) $\rightarrow$ renuevan **`571` ($50,85\%$)**, más del doble que en baja asistencia.
  * **3. Alta asistencia (`≥ 6` visitas en 21 días, $\ge 2$/semana):** `1.423` contratos ($37,88\%$ del total) $\rightarrow$ renuevan **`1.119` ($78,64\%$)**, multiplicando por **$3,57\times$** la retención del tramo bajo ($+56,59\text{ pp}$).

---
### 2. La Primera Reserva: El Catalizador de las Clases Colectivas (Primeros 14 días)
Medimos el impacto sobre la renovación de reservar al menos una clase dirigida (`v_reservas`) durante las dos primeras semanas de contrato (`<= 14 días`).

In [3]:
q_reserva = '''
WITH vencidas AS (
    SELECT *
    FROM v_suscripciones
    WHERE TRY_CAST(fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renov AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
),
res_14d AS (
    SELECT 
        cr.suscripcion_id,
        COUNT(r.reserva_id) AS n_reservas
    FROM con_renov cr
    LEFT JOIN v_reservas r
        ON cr.socio_id = r.socio_id
       AND TRY_CAST(r.fecha_reserva AS DATE) >= TRY_CAST(cr.fecha_inicio AS DATE)
       AND TRY_CAST(r.fecha_reserva AS DATE) <= TRY_CAST(cr.fecha_inicio AS DATE) + INTERVAL 14 DAY
    GROUP BY cr.suscripcion_id
)
SELECT 
    CASE WHEN n_reservas > 0 THEN '1. Reservó clase (<=14d)' ELSE '2. No reservó clase' END AS segmento_reserva,
    COUNT(*) AS total_contratos,
    ROUND(AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END) * 100, 2) AS tasa_renovacion_pct
FROM con_renov
JOIN res_14d USING (suscripcion_id)
GROUP BY segmento_reserva
ORDER BY segmento_reserva;
'''
df_reserva = con.execute(q_reserva).df()
print('=== Impacto de la Reserva Temprana en la Renovación ===')
display(df_reserva)


=== Impacto de la Reserva Temprana en la Renovación ===


,segmento_reserva,total_contratos,tasa_renovacion_pct
0,1. Reservó clase (<=14d),1572,73.28
1,2. No reservó clase,2185,36.84


**Análisis del Catalizador de Clases Colectivas (`73,28%` vs `36,84%`):**
* Los socios que **reservan al menos una clase colectiva en sus primeros 14 días** (`1.572` contratos, el $41,84\%$) alcanzan una **tasa de renovación del $73,28\%$**, exactamente **el doble ($2,0\times$, $+36,44\text{ pp}$)** que quienes no reservan ninguna clase (`2.185` contratos, **$36,84\%$ de renovación**).
* Además, el **$90,72\%$** de los socios que logran alta asistencia temprana (`≥ 6` visitas) habían reservado clase colectiva en sus primeras dos semanas: la clase dirigida con monitor y grupo actúa como el ancla social del hábito.

---
### 3. El Control de Robustez: Demolición del Mito del Plan (Control de Confusión Hábito vs. Plan)
Cruzamos el **Tramo de Asistencia Temprana (21 días)** con el **Tipo de Plan (`anual` vs `mensual`)** para responder a la pregunta decisiva del Comité: **a igualdad de hábito inicial, ¿importa el plan contratado?**

In [4]:
q_control = '''
WITH vencidas AS (
    SELECT *
    FROM v_suscripciones
    WHERE TRY_CAST(fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renov AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
),
checkins_21d AS (
    SELECT 
        cr.suscripcion_id,
        COUNT(c.checkin_id) AS n_checkins_21d
    FROM con_renov cr
    LEFT JOIN v_checkins c 
        ON cr.socio_id = c.socio_id
       AND TRY_CAST(c.fecha_hora AS DATE) >= TRY_CAST(cr.fecha_inicio AS DATE)
       AND TRY_CAST(c.fecha_hora AS DATE) <= TRY_CAST(cr.fecha_inicio AS DATE) + INTERVAL 21 DAY
    GROUP BY cr.suscripcion_id
),
datos_completos AS (
    SELECT 
        cr.*,
        COALESCE(ch.n_checkins_21d, 0) AS checkins_21d,
        CASE 
            WHEN COALESCE(ch.n_checkins_21d, 0) <= 2 THEN '1. Baja (0-2)'
            WHEN COALESCE(ch.n_checkins_21d, 0) <= 5 THEN '2. Media (3-5)'
            ELSE '3. Alta (6+)'
        END AS tramo_asistencia
    FROM con_renov cr
    JOIN checkins_21d ch USING (suscripcion_id)
)
SELECT 
    tramo_asistencia,
    tipo_plan,
    COUNT(*) AS total_contratos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY tipo_plan), 1) AS pct_distribucion_en_plan,
    ROUND(AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END) * 100, 2) AS tasa_renovacion_pct
FROM datos_completos
GROUP BY tramo_asistencia, tipo_plan
ORDER BY tramo_asistencia, tipo_plan;
'''
df_control = con.execute(q_control).df()
print('=== Control de Robustez: Tramo de Asistencia x Tipo de Plan ===')
display(df_control)


=== Control de Robustez: Tramo de Asistencia x Tipo de Plan ===


,tramo_asistencia,tipo_plan,total_contratos,pct_distribucion_en_plan,tasa_renovacion_pct
0,1. Baja (0-2),anual,789,51.6,22.56
1,1. Baja (0-2),mensual,422,18.9,21.09
2,2. Media (3-5),anual,434,28.4,48.62
3,2. Media (3-5),mensual,689,30.9,52.25
4,3. Alta (6+),anual,307,20.1,78.18
5,3. Alta (6+),mensual,1116,50.1,78.76


### Conclusiones Día 4 (jueves) — El Hallazgo Central:

1. **La Escalera del Hábito:** La asistencia en los primeros 21 días es un predictor fulminante de renovación:
   * **0-2 visitas:** **22,05%** de renovación.
   * **3-5 visitas:** **50,85%** de renovación.
   * **6+ visitas:** **78,64%** de renovación (¡casi 4 veces más retención!).
2. **El Catalizador de Clases Colectivas:** Reservar al menos una clase en las primeras 2 semanas eleva la renovación del **36,8% al 73,3%**.
3. **La Demolición del Mito (Control de Robustez):**
   * En el tramo de alta asistencia ($\ge 6$), **el Anual renueva al 78,18% y el Mensual al 78,76%** (idénticos).
   * En el tramo de baja asistencia ($0-2$), **el Anual renueva al 22,56% y el Mensual al 21,09%** (idénticos).
4. **La Explicación de la Grieta:**
   * El **51,6%** de los socios anuales caen en el tramo de baja asistencia, frente a sólo el **18,9%** de los mensuales.
   * El prepago anual quita la urgencia cotidiana de «amortizar la cuota del mes», permitiendo que el socio desconecte sin dolor inmediato.
   * **La Palanca Real:** El plan no es el veneno; el abandono del onboarding temprano es el asesino silencioso.
